# ⚡ Spark Streaming & Real-Time Processing — Hands-On Lab

**Format:** Google Colab hands-on notebook · **Level:** Intermediate · **Duration:** ~2.5–3 hours (with discussion)

### What you will learn
| # | Topic | What you will build |
|---|-------|---------------------|
| 1 | Spark Streaming Architecture | See the micro-batch engine in action (driver, receivers, batch interval) |
| 2 | DStreams (Discretized Streams) | Prove that a DStream is "a sequence of RDDs" |
| 3 | Basic Transformations & Actions | `map`, `flatMap`, `filter`, `reduceByKey`, `countByValue`, `transform`, `union`, `pprint`, `foreachRDD`, `saveAsTextFiles` |
| 4 | Windowed Operations | Sliding & tumbling windows, `countByWindow`, `reduceByKeyAndWindow` (with/without inverse) |
| 5 | Stateful Operations | `updateStateByKey` running totals, session state, state expiry |
| 6 | **Case Study** | Real-time fraud detection pipeline for financial transactions |
| 7 | Bonus | The same case study in **Structured Streaming** (the modern API) |

### How to run
1. `Runtime → Run all` (or run cells top to bottom with `Shift + Enter`).
2. Streaming cells run for a fixed number of seconds and then stop by themselves — wait for each one to finish.
3. Everything is self-contained: no Kafka, sockets, or external data needed. We simulate live data with a **queue stream**.

> 💡 **Why a queue stream?** In production, data arrives from Kafka, Kinesis, sockets or files. In a classroom notebook we push
> pre-built RDDs into a queue, and Spark consumes **one RDD per batch** — exactly like a live source, but repeatable.

---
## 0. Environment Setup
We install **PySpark 3.5.3** (DStreams are fully supported there) and make sure a compatible **Java (8/11/17)** is available.

In [ ]:
# ------------------------------------------------------------
# STEP 0.1 : Install Java (if needed) and PySpark 3.5.3
# ------------------------------------------------------------
import os, re, sys, shutil, subprocess
from importlib import metadata

def java_major_version():
    """Return the major Java version on PATH (e.g. 11, 17), or None if Java is missing."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        m = re.search(r'version "(\d+)(?:\.(\d+))?', out)
        major = int(m.group(1))
        return int(m.group(2)) if major == 1 else major   # "1.8" -> 8
    except Exception:
        return None

# Spark 3.5 officially supports Java 8 / 11 / 17
if java_major_version() not in (8, 11, 17):
    print("Installing OpenJDK 17 (takes ~30 seconds)...")
    subprocess.run("apt-get update -qq && apt-get install -y -qq openjdk-17-jdk-headless > /dev/null",
                   shell=True, check=True)
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

# Install PySpark 3.5.3 unless it is already present (we check WITHOUT importing it)
try:
    installed = metadata.version("pyspark")
except metadata.PackageNotFoundError:
    installed = None
if installed != "3.5.3":
    print(f"Found pyspark={installed}; installing pyspark==3.5.3 ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark==3.5.3"])

print("Java version :", java_major_version())
print("PySpark      :", metadata.version("pyspark"))

Java version : 17
PySpark      : 3.5.3


In [ ]:
# ------------------------------------------------------------
# STEP 0.2 : Imports and folders for this lab
# ------------------------------------------------------------
import time, random, json, glob
from datetime import datetime, timedelta
from operator import add, sub

import pandas as pd
import matplotlib.pyplot as plt

from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.streaming import StreamingContext   # <- the DStream API lives here

# Work inside /content on Colab, or the current folder anywhere else
BASE_DIR = "/content/spark_streaming_lab" if os.path.isdir("/content") else os.path.abspath("spark_streaming_lab")
shutil.rmtree(BASE_DIR, ignore_errors=True)        # fresh start every time you run the notebook
os.makedirs(BASE_DIR, exist_ok=True)
CHECKPOINT_DIR = os.path.join(BASE_DIR, "checkpoints")
OUTPUT_DIR     = os.path.join(BASE_DIR, "output")
print("Lab folder:", BASE_DIR)

Lab folder: /content/spark_streaming_lab


In [ ]:
# ------------------------------------------------------------
# STEP 0.3 : Start Spark (one SparkContext for the whole lab)
# ------------------------------------------------------------
conf = (SparkConf()
        .setMaster("local[*]")                         # use all Colab CPU cores
        .setAppName("SparkStreamingLab")
        .set("spark.default.parallelism", "2")         # small data -> few partitions = faster
        .set("spark.sql.shuffle.partitions", "2")
        .set("spark.ui.showConsoleProgress", "false")  # hide progress bars in cell output
        .set("spark.streaming.gracefulStopTimeout", "60s"))  # let in-flight batches finish on stop

spark = SparkSession.builder.config(conf=conf).getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("ERROR")                                # keep the notebook output clean

# Stateful and inverse-window operations MUST checkpoint their state RDDs.
# We set the checkpoint folder on the SparkContext (works with queue streams).
# In production you would call ssc.checkpoint(<HDFS/S3 path>) instead (see mentor notes).
sc.setCheckpointDir(CHECKPOINT_DIR)

print("Spark version     :", spark.version)
print("Master            :", sc.master)
print("Default partitions:", sc.defaultParallelism)

Spark version     : 3.5.3
Master            : local[*]
Default partitions: 2


In [ ]:
# ------------------------------------------------------------
# STEP 0.4 : Helper functions used by every streaming demo
# ------------------------------------------------------------
def make_queue_stream(ssc, batches, num_slices=2):
    """
    Simulate a live source.
    `batches` is a list of Python lists; each inner list becomes ONE micro-batch (one RDD).
    oneAtATime=True -> Spark takes exactly one RDD from the queue per batch interval.
    """
    rdd_queue = [sc.parallelize(batch, num_slices) for batch in batches]
    return ssc.queueStream(rdd_queue, oneAtATime=True)


def collect_into(store):
    """
    Returns a function for foreachRDD that saves (batch_time, records) into a Python list.
    foreachRDD runs on the DRIVER once per batch, so we can safely append to a list.
    """
    def _save(batch_time, rdd):
        store.append((batch_time.strftime("%H:%M:%S"), rdd.collect()))
    return _save


def run_stream(ssc, n_batches, batch_seconds=1, extra_seconds=4):
    """
    Start the StreamingContext, let it run long enough to consume all queued batches,
    then stop it gracefully WITHOUT stopping the SparkContext (so we can start the next demo).
    """
    ssc.start()
    ssc.awaitTerminationOrTimeout(n_batches * batch_seconds + extra_seconds)
    ssc.stop(stopSparkContext=False, stopGraceFully=True)
    print(f"✅ Stream stopped after ~{n_batches * batch_seconds + extra_seconds} seconds")


def show_batches(store, title, limit=None):
    """Pretty-print what we collected, batch by batch."""
    print(f"\n=== {title} ===")
    for i, (t, records) in enumerate(store, start=1):
        if limit and i > limit:
            break
        try:
            records = sorted(records)
        except TypeError:
            pass
        print(f"Batch {i:>2} [{t}] -> {records}")

print("Helpers ready ✔")

Helpers ready ✔


---
## 1. Spark Streaming Architecture

```
  Live data sources                    SPARK STREAMING (driver)                    Sinks
 ┌──────────────┐     ┌───────────────────────────────────────────────┐     ┌───────────────┐
 │ Kafka        │     │  Receiver / Direct reader                     │     │ Databases     │
 │ Kinesis      │ ──▶ │      │  chops the stream every batch interval │ ──▶ │ Dashboards    │
 │ Sockets/Files│     │      ▼                                        │     │ HDFS / S3     │
 └──────────────┘     │  DStream = RDD@t1, RDD@t2, RDD@t3 ...         │     │ Alerts (SMS)  │
                      │      │  JobGenerator turns each RDD into jobs │     └───────────────┘
                      │      ▼                                        │
                      │  Spark Core engine (executors run tasks)      │
                      └───────────────────────────────────────────────┘
```

**Key components**
* **StreamingContext** – entry point; owns the *batch interval* (e.g. 1 second).
* **Receiver / Input DStream** – pulls data from the source and stores it as blocks.
* **JobGenerator** – every batch interval, turns the latest block(s) into an RDD and schedules Spark jobs.
* **Executors** – run the tasks exactly like a normal batch Spark job.
* **Output operations** – push results out (print, save, write to DB).

👉 **Micro-batching:** Spark Streaming is *not* record-at-a-time. It collects data for `batch interval` seconds and processes it as a small batch job.

Let's watch the micro-batch clock tick.

In [ ]:
# ------------------------------------------------------------
# DEMO 1 : Watch micro-batches being created every batch interval
# ------------------------------------------------------------
BATCH_INTERVAL = 1                      # seconds
ssc = StreamingContext(sc, BATCH_INTERVAL)

# 4 "seconds" of incoming events (each inner list = data that arrived during one interval)
incoming = [
    ["evt-1", "evt-2"],
    ["evt-3"],
    [],                                 # no data arrived in this interval -> an EMPTY batch
    ["evt-4", "evt-5", "evt-6"],
]
events = make_queue_stream(ssc, incoming)

def describe_batch(batch_time, rdd):
    # This function runs on the DRIVER every batch interval
    print(f"[{batch_time.strftime('%H:%M:%S')}] new micro-batch -> "
          f"RDD id={rdd.id()}, partitions={rdd.getNumPartitions()}, records={rdd.count()}")

events.foreachRDD(describe_batch)       # an OUTPUT operation is mandatory, otherwise start() fails

run_stream(ssc, n_batches=len(incoming))

/usr/local/lib/python3.13/dist-packages/pyspark/streaming/context.py:72: FutureWarning: DStream is deprecated as of Spark 3.4.0. Migrate to Structured Streaming.
  warnings.warn(


[03:37:10] new micro-batch -> RDD id=0, partitions=2, records=2
[03:37:11] new micro-batch -> RDD id=1, partitions=2, records=1
[03:37:12] new micro-batch -> RDD id=2, partitions=2, records=0
[03:37:13] new micro-batch -> RDD id=3, partitions=2, records=3
[03:37:14] new micro-batch -> RDD id=4, partitions=1, records=0
[03:37:15] new micro-batch -> RDD id=4, partitions=1, records=0
[03:37:16] new micro-batch -> RDD id=4, partitions=1, records=0
[03:37:17] new micro-batch -> RDD id=4, partitions=1, records=0
[03:37:18] new micro-batch -> RDD id=4, partitions=1, records=0
[03:37:19] new micro-batch -> RDD id=4, partitions=1, records=0
✅ Stream stopped after ~8 seconds


**Observe:**
* A new RDD (new `id`) is created **every second**, even when no data arrives (empty batch).
* Batches continue after the queue is empty — a real stream never "ends"; we stop it ourselves.
* After `ssc.stop()`, a StreamingContext **cannot be restarted** — that is why each demo creates a new one.

---
## 2. DStreams (Discretized Streams)

A **DStream** is a continuous stream of data represented internally as a **sequence of RDDs**, one RDD per batch interval.

```
DStream "lines"   : | RDD @ t=1s | RDD @ t=2s | RDD @ t=3s | ...
                         │ map()        │ map()        │ map()
DStream "words"   : | RDD @ t=1s | RDD @ t=2s | RDD @ t=3s | ...
```
Any operation on a DStream is translated into the same operation on **every underlying RDD**.

**Types of input DStreams:** `socketTextStream`, `textFileStream` (folder monitoring), Kafka (`spark-streaming-kafka`), and `queueStream` (testing).

In [ ]:
# ------------------------------------------------------------
# DEMO 2 : A DStream really is a sequence of RDDs
# ------------------------------------------------------------
ssc = StreamingContext(sc, 1)

sensor_batches = [
    [21.5, 22.0, 22.4, 23.1],          # batch 1 : temperatures (°C)
    [24.0, 24.6],                      # batch 2
    [25.2, 26.8, 27.5],                # batch 3
]
temps = make_queue_stream(ssc, sensor_batches, num_slices=2)

# A transformation on a DStream returns a NEW DStream (lazy - nothing runs yet)
temps_f = temps.map(lambda c: round(c * 9 / 5 + 32, 1))   # Celsius -> Fahrenheit
print("type(temps)   :", type(temps).__name__)
print("type(temps_f) :", type(temps_f).__name__)

def inspect_rdd(batch_time, rdd):
    # glom() groups elements by partition so we can see how the batch is split
    print(f"[{batch_time.strftime('%H:%M:%S')}] RDD id={rdd.id()} "
          f"partitions={rdd.glom().collect()}")

temps_f.foreachRDD(inspect_rdd)
run_stream(ssc, n_batches=len(sensor_batches))

type(temps)   : DStream
type(temps_f) : TransformedDStream
[03:37:53] RDD id=22 partitions=[[70.7, 71.6], [72.3, 73.6]]
[03:37:54] RDD id=24 partitions=[[75.2], [76.3]]
[03:37:55] RDD id=26 partitions=[[77.4], [80.2, 81.5]]
[03:37:56] RDD id=28 partitions=[[]]
[03:37:57] RDD id=30 partitions=[[]]
[03:37:58] RDD id=32 partitions=[[]]
[03:37:59] RDD id=34 partitions=[[]]
[03:38:00] RDD id=36 partitions=[[]]
[03:38:01] RDD id=38 partitions=[[]]
✅ Stream stopped after ~7 seconds


**Observe:** each batch arrives as its own RDD with its own partitions; `map()` was applied to every RDD separately.
DStreams inherit RDD properties: **immutable, partitioned, fault-tolerant through lineage**.

---
## 3. Basic Transformations and Actions on DStreams

| Category | Operations |
|----------|-----------|
| Element-wise | `map`, `flatMap`, `filter`, `repartition` |
| Aggregations (per batch) | `count`, `reduce`, `countByValue`, `reduceByKey`, `groupByKey` |
| Combine streams | `union`, `join`, `cogroup` |
| Escape hatch | `transform` — run ANY RDD-to-RDD function on every batch |
| **Output operations (actions)** | `pprint`, `foreachRDD`, `saveAsTextFiles` |

⚠️ All per-batch aggregations work **only inside one batch** — they forget previous batches (Sections 4 & 5 fix that).

In [ ]:
# ------------------------------------------------------------
# DEMO 3.1 : map / flatMap / filter / reduceByKey  (streaming word count on app logs)
# ------------------------------------------------------------
ssc = StreamingContext(sc, 1)

log_batches = [
    ["INFO user login success", "ERROR payment gateway timeout", "INFO user logout"],
    ["WARN high latency payment api", "ERROR payment declined", "ERROR db connection lost"],
    ["INFO user login success", "INFO payment success"],
]
lines = make_queue_stream(ssc, log_batches)

STOP_WORDS = {"user", "api"}

word_counts = (lines
               .flatMap(lambda line: line.lower().split())        # 1 line -> many words
               .filter(lambda w: w not in STOP_WORDS)             # drop noise words
               .map(lambda w: (w, 1))                             # (word, 1) pairs
               .reduceByKey(add))                                 # sum per word, per batch

error_lines = lines.filter(lambda line: line.startswith("ERROR"))  # only error lines

wc_store, err_store = [], []
word_counts.foreachRDD(collect_into(wc_store))
error_lines.foreachRDD(collect_into(err_store))

run_stream(ssc, n_batches=len(log_batches))
show_batches(wc_store, "Word count per batch", limit=3)
show_batches(err_store, "ERROR lines per batch", limit=3)

✅ Stream stopped after ~7 seconds

=== Word count per batch ===
Batch  1 [03:38:27] -> [('error', 1), ('gateway', 1), ('info', 2), ('login', 1), ('logout', 1), ('payment', 1), ('success', 1), ('timeout', 1)]
Batch  2 [03:38:28] -> [('connection', 1), ('db', 1), ('declined', 1), ('error', 2), ('high', 1), ('latency', 1), ('lost', 1), ('payment', 2), ('warn', 1)]
Batch  3 [03:38:29] -> [('info', 2), ('login', 1), ('payment', 1), ('success', 2)]

=== ERROR lines per batch ===
Batch  1 [03:38:27] -> ['ERROR payment gateway timeout']
Batch  2 [03:38:28] -> ['ERROR db connection lost', 'ERROR payment declined']
Batch  3 [03:38:29] -> []


In [ ]:
# ------------------------------------------------------------
# DEMO 3.2 : count / countByValue / reduce  (quick per-batch metrics)
# ------------------------------------------------------------
ssc = StreamingContext(sc, 1)
lines = make_queue_stream(ssc, log_batches)

levels       = lines.map(lambda line: line.split()[0])     # INFO / WARN / ERROR
total_lines  = lines.count()                               # DStream of one number per batch
level_counts = levels.countByValue()                       # DStream of (level, count)
longest_line = lines.map(len).reduce(max)                  # reduce -> one value per batch

s1, s2, s3 = [], [], []
total_lines.foreachRDD(collect_into(s1))
level_counts.foreachRDD(collect_into(s2))
longest_line.foreachRDD(collect_into(s3))

run_stream(ssc, n_batches=len(log_batches))
show_batches(s1, "count()", limit=3)
show_batches(s2, "countByValue() of log level", limit=3)
show_batches(s3, "reduce(max) of line length", limit=3)

✅ Stream stopped after ~7 seconds

=== count() ===
Batch  1 [03:38:55] -> [3]
Batch  2 [03:38:56] -> [3]
Batch  3 [03:38:57] -> [2]

=== countByValue() of log level ===
Batch  1 [03:38:55] -> [('ERROR', 1), ('INFO', 2)]
Batch  2 [03:38:56] -> [('ERROR', 2), ('WARN', 1)]
Batch  3 [03:38:57] -> [('INFO', 2)]

=== reduce(max) of line length ===
Batch  1 [03:38:55] -> [29]
Batch  2 [03:38:56] -> [29]
Batch  3 [03:38:57] -> [23]


In [ ]:
# ------------------------------------------------------------
# DEMO 3.3 : transform()  -> join every batch with STATIC reference data + sort
# ------------------------------------------------------------
ssc = StreamingContext(sc, 1)

# Static lookup table (could be loaded from a DB / CSV once at startup)
error_catalog = sc.parallelize([
    ("E101", "Payment gateway timeout"),
    ("E202", "Card declined by bank"),
    ("E303", "Database connection lost"),
])

error_event_batches = [
    [("E101", "srv-1"), ("E202", "srv-2"), ("E101", "srv-3")],
    [("E303", "srv-1"), ("E101", "srv-2")],
    [("E202", "srv-4")],
]
error_events = make_queue_stream(ssc, error_event_batches)

# transform gives us the raw RDD of each batch -> we can use ANY RDD API (join, sortBy, ...)
enriched_errors = error_events.transform(
    lambda rdd: rdd.join(error_catalog)                       # (code, (server, description))
                   .map(lambda kv: (kv[0], kv[1][1], kv[1][0]))
                   .sortBy(lambda rec: rec[0])                # sort inside the batch
)

store = []
enriched_errors.foreachRDD(collect_into(store))
run_stream(ssc, n_batches=len(error_event_batches))
show_batches(store, "Errors enriched with description (transform + join)")

✅ Stream stopped after ~7 seconds

=== Errors enriched with description (transform + join) ===
Batch  1 [03:39:17] -> [('E101', 'Payment gateway timeout', 'srv-1'), ('E101', 'Payment gateway timeout', 'srv-3'), ('E202', 'Card declined by bank', 'srv-2')]
Batch  2 [03:39:18] -> [('E101', 'Payment gateway timeout', 'srv-2'), ('E303', 'Database connection lost', 'srv-1')]
Batch  3 [03:39:19] -> [('E202', 'Card declined by bank', 'srv-4')]
Batch  4 [03:39:20] -> []
Batch  5 [03:39:21] -> []
Batch  6 [03:39:22] -> []
Batch  7 [03:39:23] -> []
Batch  8 [03:39:24] -> []
Batch  9 [03:39:25] -> []


In [ ]:
# ------------------------------------------------------------
# DEMO 3.4 : union()  -> merge two live streams into one
# ------------------------------------------------------------
ssc = StreamingContext(sc, 1)

upi_batches  = [[("UPI", 250.0), ("UPI", 1200.0)], [("UPI", 75.0)], [("UPI", 999.0)]]
card_batches = [[("CARD", 4500.0)], [("CARD", 300.0), ("CARD", 15000.0)], []]

upi  = make_queue_stream(ssc, upi_batches)
card = make_queue_stream(ssc, card_batches)

all_payments = upi.union(card)                    # both DStreams must have the same batch interval
amount_by_channel = all_payments.reduceByKey(add)

store = []
amount_by_channel.foreachRDD(collect_into(store))
run_stream(ssc, n_batches=3)
show_batches(store, "Total amount per payment channel (UPI ∪ CARD)", limit=3)

✅ Stream stopped after ~7 seconds

=== Total amount per payment channel (UPI ∪ CARD) ===
Batch  1 [03:51:10] -> [('CARD', 4500.0), ('UPI', 1450.0)]
Batch  2 [03:51:11] -> [('CARD', 15300.0), ('UPI', 75.0)]
Batch  3 [03:51:12] -> [('UPI', 999.0)]


In [ ]:
# ------------------------------------------------------------
# DEMO 3.5 : Output operations  -> pprint, saveAsTextFiles, foreachRDD (to a DataFrame)
# ------------------------------------------------------------
ssc = StreamingContext(sc, 1)

order_batches = [
    [("ORD-1", "Mumbai", 1200.0), ("ORD-2", "Delhi", 560.0)],
    [("ORD-3", "Pune", 3400.0)],
    [("ORD-4", "Mumbai", 80.0), ("ORD-5", "Chennai", 2300.0)],
]
orders = make_queue_stream(ssc, order_batches)

# (a) pprint: prints the first N elements of every batch on the driver (great for debugging)
orders.pprint(3)

# (b) saveAsTextFiles: writes ONE folder per batch named <prefix>-<batch time in ms>.<suffix>
orders_prefix = os.path.join(OUTPUT_DIR, "orders")
orders.map(lambda o: f"{o[0]},{o[1]},{o[2]}").saveAsTextFiles(orders_prefix, "csv")

# (c) foreachRDD: the most flexible sink -> convert each batch to a DataFrame and write it anywhere
df_batches = []
def to_dataframe(batch_time, rdd):
    if rdd.isEmpty():                             # always guard against empty batches
        return
    df = spark.createDataFrame(rdd, ["order_id", "city", "amount"])
    df_batches.append(df.toPandas().assign(batch_time=batch_time.strftime("%H:%M:%S")))

orders.foreachRDD(to_dataframe)

run_stream(ssc, n_batches=len(order_batches))

print("\nFolders written by saveAsTextFiles (one per batch):")
for folder in sorted(glob.glob(orders_prefix + "-*"))[:5]:
    print("  ", os.path.basename(folder))

print("\nAll batches collected through foreachRDD -> DataFrame:")
pd.concat(df_batches, ignore_index=True)

**Takeaways**
* Transformations are **lazy**; nothing runs until `ssc.start()` and only if at least one **output operation** exists.
* `foreachRDD` runs its function on the **driver**, but the RDD operations inside it run on the **executors**.
* `saveAsTextFiles` creates a folder for *every* batch, including empty ones — in production you usually write via `foreachRDD` instead.

---
## 4. Windowed Operations

A **window** combines the RDDs of the last *N* seconds. Two parameters (both must be multiples of the batch interval):
* **windowDuration** – how far back to look (e.g. last 3 seconds)
* **slideDuration** – how often to compute the result (e.g. every 1 second)

```
batch:      1     2     3     4     5     6
          [-----window @3-----]
                [-----window @4-----]
                      [-----window @5-----]        window=3s, slide=1s  (SLIDING)

          [--window @2--][--window @4--][--window @6--]   window=2s, slide=2s  (TUMBLING)
```

In [ ]:
# ------------------------------------------------------------
# DEMO 4.1 : window(), countByWindow(), reduceByKeyAndWindow() (no inverse)
# ------------------------------------------------------------
ssc = StreamingContext(sc, 1)

# Page clicks arriving every second on an e-commerce site
click_batches = [
    ["home", "cart"],                 # batch 1
    ["home"],                         # batch 2
    ["cart", "checkout"],             # batch 3
    ["home", "home", "cart"],         # batch 4
    [],                               # batch 5
    ["checkout"],                     # batch 6
]
clicks = make_queue_stream(ssc, click_batches)

WINDOW, SLIDE = 3, 1                                                  # last 3 s, every 1 s

windowed_clicks  = clicks.window(WINDOW, SLIDE)                       # raw contents of the window
clicks_in_window = clicks.countByWindow(WINDOW, SLIDE)                # total clicks in window
page_counts_win  = (clicks.map(lambda p: (p, 1))
                          .reduceByKeyAndWindow(add, None, WINDOW, SLIDE))  # recompute window each time

s_win, s_cnt, s_page = [], [], []
windowed_clicks.foreachRDD(collect_into(s_win))
clicks_in_window.foreachRDD(collect_into(s_cnt))
page_counts_win.foreachRDD(collect_into(s_page))

run_stream(ssc, n_batches=len(click_batches) + 2)
show_batches(s_win,  "window(3,1) contents", limit=8)
show_batches(s_cnt,  "countByWindow(3,1)", limit=8)
show_batches(s_page, "reduceByKeyAndWindow(add, None, 3, 1)", limit=8)

**Read the output like this:** at batch 4 the window covers batches 2, 3, 4 → `home, cart, checkout, home, home, cart` = 6 clicks.
Notice how batch 1's clicks "fall out" of the window as it slides forward.

In [ ]:
# ------------------------------------------------------------
# DEMO 4.2 : Incremental window with an INVERSE function + countByValueAndWindow
# ------------------------------------------------------------
# Without an inverse, Spark re-adds ALL batches in the window every slide.
# With an inverse, Spark does:  new_window = old_window + new_batch - batch_that_left
# This is MUCH cheaper for long windows (e.g. 1 hour window, 10 s slide). It requires checkpointing.
ssc = StreamingContext(sc, 1)
clicks = make_queue_stream(ssc, click_batches)

page_counts_inc = (clicks.map(lambda p: (p, 1))
                         .reduceByKeyAndWindow(add,            # add values entering the window
                                               sub,            # subtract values leaving the window
                                               windowDuration=WINDOW,
                                               slideDuration=SLIDE,
                                               filterFunc=lambda kv: kv[1] > 0))  # drop keys that reached 0

value_counts_win = clicks.countByValueAndWindow(WINDOW, SLIDE)   # built on the same inverse trick

s_inc, s_cbv = [], []
page_counts_inc.foreachRDD(collect_into(s_inc))
value_counts_win.foreachRDD(collect_into(s_cbv))

run_stream(ssc, n_batches=len(click_batches) + 2)
show_batches(s_inc, "reduceByKeyAndWindow(add, sub, 3, 1) - incremental", limit=8)
show_batches(s_cbv, "countByValueAndWindow(3, 1)", limit=8)

In [ ]:
# ------------------------------------------------------------
# DEMO 4.3 : Tumbling window (window == slide) -> non-overlapping 2-second buckets
# ------------------------------------------------------------
ssc = StreamingContext(sc, 1)
clicks = make_queue_stream(ssc, click_batches)

tumbling = clicks.map(lambda p: (p, 1)).reduceByKeyAndWindow(add, None, 2, 2)   # window=2s, slide=2s

s_tumble = []
tumbling.foreachRDD(collect_into(s_tumble))
run_stream(ssc, n_batches=len(click_batches) + 2)
show_batches(s_tumble, "Tumbling window(2,2): one result every 2 seconds", limit=4)

**Takeaways**
* Sliding windows overlap (a record is counted in several windows); tumbling windows do not.
* Output of a windowed DStream is produced **every slide**, not every batch.
* Use the **inverse-function** version for long windows; use `filterFunc` to remove keys whose count drops to zero.

---
## 5. Stateful Operations

Windows remember only the last *N* seconds. **Stateful** operations remember information **for the whole life of the stream**
(running totals, user sessions, account balances).

`updateStateByKey(update_fn)` calls, for **every key, every batch**:
```python
update_fn(new_values_in_this_batch: list, previous_state or None) -> new_state   # return None to delete the key
```
State is checkpointed regularly so it survives failures.

In [ ]:
# ------------------------------------------------------------
# DEMO 5.1 : Running total of sales per product (updateStateByKey)
# ------------------------------------------------------------
def running_total(new_values, current_total):
    """new_values = amounts that arrived in THIS batch for the key; current_total = state so far."""
    return (current_total or 0.0) + sum(new_values)

ssc = StreamingContext(sc, 1)
sales_batches = [
    [("laptop", 55000.0), ("mouse", 700.0)],
    [("mouse", 650.0), ("phone", 22000.0)],
    [],                                               # no sales -> state is still kept
    [("laptop", 61000.0)],
]
sales = make_queue_stream(ssc, sales_batches)

per_batch_sales = sales.reduceByKey(add)                 # forgets after each batch
running_sales   = sales.updateStateByKey(running_total)  # remembers forever

s_batch, s_state = [], []
per_batch_sales.foreachRDD(collect_into(s_batch))
running_sales.foreachRDD(collect_into(s_state))

run_stream(ssc, n_batches=len(sales_batches))
show_batches(s_batch, "reduceByKey  (per batch only)", limit=4)
show_batches(s_state, "updateStateByKey (running total)", limit=4)

In [ ]:
# ------------------------------------------------------------
# DEMO 5.2 : Rich state + state expiry -> user sessions that time out
# ------------------------------------------------------------
IDLE_TIMEOUT_BATCHES = 2          # remove a user after 2 batches with no activity

def update_session(new_events, session):
    """
    State per user = dict(clicks, pages, idle_batches)
    - new events  -> add them and reset idle counter
    - no events   -> increase idle counter; delete session (return None) after the timeout
    """
    if session is None:
        session = {"clicks": 0, "pages": [], "idle_batches": 0}
    if new_events:
        return {"clicks": session["clicks"] + len(new_events),
                "pages": session["pages"] + new_events,
                "idle_batches": 0}
    if session["idle_batches"] + 1 >= IDLE_TIMEOUT_BATCHES:
        return None                                   # session expired -> key removed from state
    return {**session, "idle_batches": session["idle_batches"] + 1}

ssc = StreamingContext(sc, 1)
activity_batches = [
    [("asha", "home"), ("ravi", "home")],
    [("asha", "product"), ("ravi", "cart")],
    [("asha", "checkout")],                           # ravi idle (1)
    [],                                               # asha idle (1), ravi idle (2) -> expired
    [("ravi", "home")],                               # ravi comes back -> NEW session
]
activity = make_queue_stream(ssc, activity_batches)
sessions = activity.updateStateByKey(update_session)

s_sess = []
sessions.foreachRDD(collect_into(s_sess))
run_stream(ssc, n_batches=len(activity_batches))
show_batches(s_sess, "Active sessions after each batch", limit=6)

**Takeaways**
* `updateStateByKey` visits **every key in state on every batch** — even keys without new data. Cost grows with the number of keys, so expire state you no longer need (return `None`).
* Stateful operations **require checkpointing** (we set `sc.setCheckpointDir` in setup).
* Scala/Java also offer `mapWithState` (only touches keys with new data + built-in timeouts); it is not available in PySpark.

---
## 6. 🏦 Case Study: Real-Time Data Processing for Financial Transactions

**Scenario:** *FinPay Bank* processes card/UPI transactions across India. The risk team wants a streaming pipeline that, **within seconds**:

1. **Ingests** raw transaction events and **rejects malformed records**.
2. **Enriches** each transaction with merchant category (broadcast lookup) and customer master data (static join).
3. Publishes **live KPIs** every batch (volume, value, rejects).
4. Raises **fraud alerts** using four rules:

| Rule | Technique | Logic |
|------|-----------|-------|
| `HIGH_VALUE` | stateless filter | single transaction ≥ ₹50,000 |
| `VELOCITY` (card testing) | **sliding window** | ≥ 5 transactions on one account in the last 6 s |
| `GEO_ANOMALY` (impossible travel) | **sliding window** | ≥ 3 different cities for one account in the last 6 s |
| `LIMIT_BREACH` | **stateful** | running spend of the account exceeds its monthly limit |

5. Maintains a **running customer profile** (stateful) and **running spend per category**.
6. Writes alerts to a **Parquet "alerts table"** for investigators.

```
raw events ─▶ parse/validate ─▶ enrich (broadcast + join) ─┬─▶ KPIs per batch ─────────────▶ dashboard
                   │                                        ├─▶ HIGH_VALUE (filter) ───┐
                   └─▶ rejected count                       ├─▶ VELOCITY (window) ─────┤
                                                            ├─▶ GEO_ANOMALY (window) ──┼─▶ alerts ─▶ Parquet
                                                            └─▶ profile (state) ─▶ LIMIT_BREACH ┘
```

In [ ]:
# ------------------------------------------------------------
# CASE STUDY STEP 1 : Reference data + synthetic transaction generator
# ------------------------------------------------------------
random.seed(42)                                     # same data every run -> reproducible results

MERCHANT_CATEGORY = {
    "Amazon": "E-Commerce", "Flipkart": "E-Commerce", "BigBasket": "Groceries",
    "Swiggy": "Food", "Zomato": "Food", "IndianOil": "Fuel", "Apollo Pharmacy": "Health",
    "Croma": "Electronics", "MakeMyTrip": "Travel", "ATM-Withdrawal": "Cash",
}
CITIES = ["Mumbai", "Delhi", "Bengaluru", "Pune", "Chennai", "Hyderabad"]

# Customer master data: account -> (customer name, home city, monthly limit in ₹)
CUSTOMERS = {
    "ACC1000": ("Aarav Shah",   "Mumbai",    100000.0),
    "ACC1001": ("Diya Verma",   "Delhi",     100000.0),
    "ACC1002": ("Kabir Rao",    "Bengaluru",  80000.0),
    "ACC1003": ("Meera Iyer",   "Pune",      100000.0),
    "ACC1004": ("Rohan Gupta",  "Chennai",    60000.0),
    "ACC1005": ("Sara Khan",    "Hyderabad",  50000.0),
    "ACC1006": ("Vikram Nair",  "Mumbai",     75000.0),
    "ACC1007": ("Ananya Das",   "Delhi",      90000.0),
}
ACCOUNTS = sorted(CUSTOMERS)

NUM_BATCHES = 16
BASE_TIME = datetime(2026, 11, 3, 10, 0, 0)
CASE_BATCH_SECONDS = 2                              # each micro-batch = 2 seconds of transactions

def fmt_ts(dt):
    return f"{dt:%Y-%m-%d %H:%M:%S}.{dt.microsecond // 1000:03d}"

def make_txn(txn_no, batch_no, offset, account, merchant, amount, city, channel):
    """Build one raw CSV event: txn_id,event_time,account_id,merchant,amount,city,channel"""
    ts = BASE_TIME + timedelta(seconds=batch_no * CASE_BATCH_SECONDS + offset)
    return f"T{txn_no:05d},{fmt_ts(ts)},{account},{merchant},{amount:.2f},{city},{channel}"

def generate_transaction_batches():
    batches, txn_no = [], 0
    normal_merchants = [m for m in MERCHANT_CATEGORY if m != "Croma"]
    for b in range(NUM_BATCHES):
        batch = []
        # --- normal traffic: 4-6 different accounts, one txn each, in their home city ---
        for i, acc in enumerate(random.sample(ACCOUNTS, random.randint(4, 6))):
            txn_no += 1
            merchant = random.choice(normal_merchants)
            amount = round(random.uniform(150, 6000), 2)
            channel = "ATM" if merchant == "ATM-Withdrawal" else random.choice(["POS", "UPI", "ONLINE"])
            batch.append(make_txn(txn_no, b, i * 0.2, acc, merchant, amount, CUSTOMERS[acc][1], channel))

        # --- injected fraud scenarios (so every rule fires at a known moment) ---
        if b == 3:   # HIGH_VALUE + later LIMIT_BREACH: one very large electronics purchase
            txn_no += 1
            batch.append(make_txn(txn_no, b, 1.5, "ACC1003", "Croma", 125000.00, "Pune", "POS"))
        if b in (5, 6):   # VELOCITY: card-testing burst of small online payments
            for k in range(4 if b == 5 else 3):
                txn_no += 1
                batch.append(make_txn(txn_no, b, 1.0 + k * 0.1, "ACC1005", "Amazon", 49.00, "Hyderabad", "ONLINE"))
        if b == 8:   # GEO_ANOMALY: same card used in far-apart cities within seconds
            for k, city in enumerate(["Mumbai", "Chennai"]):
                txn_no += 1
                batch.append(make_txn(txn_no, b, 1.2 + k * 0.1, "ACC1001", "ATM-Withdrawal", 9500.00, city, "ATM"))
        if b == 9:
            txn_no += 1
            batch.append(make_txn(txn_no, b, 1.3, "ACC1001", "ATM-Withdrawal", 9800.00, "Hyderabad", "ATM"))

        # --- dirty data that a real pipeline must survive ---
        if b == 2:
            batch.append("CORRUPTED###RECORD")                                  # garbage line
        if b == 4:
            batch.append("T99998,2026-11-03 10:00:08.000,ACC1002,Swiggy,-450.00,Bengaluru,UPI")  # negative amount
        if b == 10:
            batch.append("T99999,2026-11-03 10:00:20.000,ACC1004,Zomato,,Chennai")           # missing fields
        batches.append(batch)
    return batches

txn_batches = generate_transaction_batches()
print(f"Generated {NUM_BATCHES} batches, {sum(len(b) for b in txn_batches)} raw events")
print("Sample raw events from batch 4:")
for line in txn_batches[3][:4]:
    print("  ", line)

In [ ]:
# ------------------------------------------------------------
# CASE STUDY STEP 2 : Parsing, enrichment and state functions (plain Python, easy to unit test)
# ------------------------------------------------------------
def parse_txn(line):
    """Parse one CSV line into a dict. Return None for malformed / invalid records."""
    try:
        parts = [p.strip() for p in line.split(",")]
        if len(parts) != 7:
            return None
        txn_id, event_time, account_id, merchant, amount, city, channel = parts
        amount = float(amount)
        if amount <= 0 or not account_id.startswith("ACC"):
            return None
        return {"txn_id": txn_id, "event_time": event_time, "account_id": account_id,
                "merchant": merchant, "amount": amount, "city": city, "channel": channel}
    except (ValueError, AttributeError):
        return None


def merge_customer(pair):
    """(account_id, (txn_dict, (name, home_city, limit))) -> one flat enriched dict."""
    _, (txn, (name, home_city, monthly_limit)) = pair
    return {**txn, "customer": name, "home_city": home_city, "monthly_limit": monthly_limit}


def update_profile(new_txns, profile):
    """Stateful running profile per account (called for every account, every batch)."""
    if profile is None:
        profile = {"txn_count": 0, "total_spend": 0.0, "max_txn": 0.0,
                   "cities": [], "monthly_limit": None}
    if not new_txns:
        return profile                                  # no activity: keep state unchanged
    cities = set(profile["cities"]) | {t["city"] for t in new_txns}
    return {
        "txn_count":     profile["txn_count"] + len(new_txns),
        "total_spend":   round(profile["total_spend"] + sum(t["amount"] for t in new_txns), 2),
        "max_txn":       max([profile["max_txn"]] + [t["amount"] for t in new_txns]),
        "cities":        sorted(cities),
        "monthly_limit": new_txns[0]["monthly_limit"],
    }


def update_category_total(new_amounts, total):
    return round((total or 0.0) + sum(new_amounts), 2)

# Quick unit tests - always test streaming logic as plain functions first!
assert parse_txn("CORRUPTED###RECORD") is None
assert parse_txn("T1,2026-11-03 10:00:00.000,ACC1002,Swiggy,-450.00,Bengaluru,UPI") is None
assert parse_txn("T1,2026-11-03 10:00:00.000,ACC1002,Swiggy,450.00,Bengaluru,UPI")["amount"] == 450.0
p = update_profile([{"amount": 100.0, "city": "Pune", "monthly_limit": 1000.0}], None)
p = update_profile([{"amount": 50.0, "city": "Delhi", "monthly_limit": 1000.0}], p)
assert p["txn_count"] == 2 and p["total_spend"] == 150.0 and p["cities"] == ["Delhi", "Pune"]
print("Unit tests passed ✔")

In [ ]:
# ------------------------------------------------------------
# CASE STUDY STEP 3 : Build and run the streaming pipeline
# ------------------------------------------------------------
HIGH_VALUE_THRESHOLD = 50000.0
VELOCITY_THRESHOLD   = 5          # txns per account in the window
GEO_CITY_THRESHOLD   = 3          # distinct cities per account in the window
WINDOW_SEC, SLIDE_SEC = 6, 2      # look at the last 6 seconds, re-evaluate every 2 seconds
ALERTS_PATH = os.path.join(OUTPUT_DIR, "fraud_alerts_parquet")

ssc = StreamingContext(sc, CASE_BATCH_SECONDS)

# ---- Reference data: broadcast small dicts, use RDDs for joinable master data ----
merchant_bc      = sc.broadcast(MERCHANT_CATEGORY)                    # shipped once to every executor
customer_rdd     = sc.parallelize([(acc, info) for acc, info in CUSTOMERS.items()]).cache()

# ---- 1) Ingest + validate ----
raw     = make_queue_stream(ssc, txn_batches)
parsed  = raw.map(parse_txn)
valid   = parsed.filter(lambda t: t is not None)
rejects = parsed.filter(lambda t: t is None).count()                  # one number per batch

# ---- 2) Enrich: broadcast lookup + transform-join with customer master ----
enriched = (valid
            .map(lambda t: {**t, "category": merchant_bc.value.get(t["merchant"], "Other")})
            .map(lambda t: (t["account_id"], t))
            .transform(lambda rdd: rdd.join(customer_rdd))
            .map(merge_customer))
enriched.cache()                                                      # reused by many branches below

# ---- 3) Rule HIGH_VALUE (stateless) ----
high_value_alerts = (enriched
    .filter(lambda t: t["amount"] >= HIGH_VALUE_THRESHOLD)
    .map(lambda t: (t["account_id"], "HIGH_VALUE",
                    f"Rs {t['amount']:,.0f} at {t['merchant']} ({t['city']})", t["amount"])))

# ---- 4) Rule VELOCITY (sliding window with inverse function) ----
velocity_alerts = (enriched
    .map(lambda t: (t["account_id"], 1))
    .reduceByKeyAndWindow(add, sub, windowDuration=WINDOW_SEC, slideDuration=SLIDE_SEC,
                          filterFunc=lambda kv: kv[1] > 0)
    .filter(lambda kv: kv[1] >= VELOCITY_THRESHOLD)
    .map(lambda kv: (kv[0], "VELOCITY", f"{kv[1]} txns in last {WINDOW_SEC}s", float(kv[1]))))

# ---- 5) Rule GEO_ANOMALY (window + set union per account) ----
geo_alerts = (enriched
    .map(lambda t: (t["account_id"], {t["city"]}))
    .window(WINDOW_SEC, SLIDE_SEC)
    .reduceByKey(lambda a, b: a | b)
    .filter(lambda kv: len(kv[1]) >= GEO_CITY_THRESHOLD)
    .map(lambda kv: (kv[0], "GEO_ANOMALY",
                     f"{len(kv[1])} cities in {WINDOW_SEC}s: {', '.join(sorted(kv[1]))}", float(len(kv[1])))))

# ---- 6) Stateful customer profile + Rule LIMIT_BREACH ----
profiles = enriched.map(lambda t: (t["account_id"], t)).updateStateByKey(update_profile)
limit_alerts = (profiles
    .filter(lambda kv: kv[1]["monthly_limit"] is not None
                       and kv[1]["total_spend"] > kv[1]["monthly_limit"])
    .map(lambda kv: (kv[0], "LIMIT_BREACH",
                     f"spent Rs {kv[1]['total_spend']:,.0f} > limit Rs {kv[1]['monthly_limit']:,.0f}",
                     kv[1]["total_spend"])))

category_totals = enriched.map(lambda t: (t["category"], t["amount"])).updateStateByKey(update_category_total)

all_alerts = high_value_alerts.union(velocity_alerts).union(geo_alerts).union(limit_alerts)

# ---------------- OUTPUT OPERATIONS (sinks) ----------------
kpi_rows, latest_profiles, latest_categories, alert_log = [], {}, {}, []
rejects_by_batch = {}

def sink_rejects(batch_time, rdd):
    vals = rdd.collect()
    rejects_by_batch[batch_time.strftime("%H:%M:%S")] = vals[0] if vals else 0

def sink_kpis(batch_time, rdd):
    t = batch_time.strftime("%H:%M:%S")
    amounts = rdd.map(lambda x: x["amount"]).collect()
    row = {"batch_time": t, "txn_count": len(amounts), "total_value": round(sum(amounts), 2),
           "avg_value": round(sum(amounts) / len(amounts), 2) if amounts else 0.0,
           "rejected": rejects_by_batch.get(t, 0)}
    kpi_rows.append(row)
    print(f"[{t}] txns={row['txn_count']:>2}  value=Rs {row['total_value']:>10,.2f}  rejected={row['rejected']}")

def sink_profiles(batch_time, rdd):
    latest_profiles.clear()
    latest_profiles.update(dict(rdd.collect()))          # latest snapshot of the state

def sink_categories(batch_time, rdd):
    latest_categories.clear()
    latest_categories.update(dict(rdd.collect()))

def sink_alerts(batch_time, rdd):
    if rdd.isEmpty():
        return
    t = batch_time.strftime("%H:%M:%S")
    rows = rdd.map(lambda a: (t, a[0], a[1], a[2], float(a[3]))).collect()
    alert_log.extend(rows)
    # Write this batch of alerts to the Parquet "alerts table" (append mode)
    (spark.createDataFrame(rows, ["batch_time", "account_id", "rule", "detail", "metric"])
          .write.mode("append").parquet(ALERTS_PATH))
    for r in rows:
        print(f"   🚨 [{t}] {r[2]:<12} {r[1]} -> {r[3]}")

# Order matters: rejects are computed before KPIs in each batch
rejects.foreachRDD(sink_rejects)
enriched.foreachRDD(sink_kpis)
profiles.foreachRDD(sink_profiles)
category_totals.foreachRDD(sink_categories)
all_alerts.foreachRDD(sink_alerts)

print("Starting the FinPay real-time pipeline...\n")
run_stream(ssc, n_batches=NUM_BATCHES, batch_seconds=CASE_BATCH_SECONDS, extra_seconds=6)

In [ ]:
# ------------------------------------------------------------
# CASE STUDY STEP 4 : Analyse the results
# ------------------------------------------------------------
kpi_df = pd.DataFrame(kpi_rows)
print("📊 Live KPIs per micro-batch")
display(kpi_df.head(NUM_BATCHES))

# Read back the Parquet alerts table with Spark SQL
alerts_sdf = spark.read.parquet(ALERTS_PATH)
alerts_sdf.createOrReplaceTempView("fraud_alerts")
print("\n🚨 Alerts by rule (raw, before de-duplication)")
spark.sql("""
    SELECT rule, COUNT(*) AS raw_alerts, COUNT(DISTINCT account_id) AS accounts
    FROM fraud_alerts GROUP BY rule ORDER BY rule
""").show(truncate=False)

# Sliding windows + state re-fire the same alert in several batches -> keep the FIRST occurrence
alerts_pdf = alerts_sdf.toPandas().sort_values(["batch_time", "rule", "account_id"])
unique_alerts = alerts_pdf.drop_duplicates(subset=["account_id", "rule"], keep="first")
print("🔎 Unique alerts for the investigation team")
display(unique_alerts.reset_index(drop=True))

In [ ]:
# ------------------------------------------------------------
# CASE STUDY STEP 5 : Final state - customer profiles and category spend
# ------------------------------------------------------------
profile_df = (pd.DataFrame.from_dict(latest_profiles, orient="index")
                .rename_axis("account_id").reset_index().sort_values("account_id"))
profile_df["limit_used_%"] = (profile_df["total_spend"] / profile_df["monthly_limit"] * 100).round(1)
print("👤 Running customer profiles (state after the last batch)")
display(profile_df[["account_id", "txn_count", "total_spend", "max_txn", "monthly_limit", "limit_used_%", "cities"]])

category_df = (pd.Series(latest_categories, name="total_spend").rename_axis("category")
                 .reset_index().sort_values("total_spend", ascending=False))

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
axes[0].plot(range(1, len(kpi_df) + 1), kpi_df["txn_count"], marker="o")
axes[0].set_title("Transactions per micro-batch"); axes[0].set_xlabel("Batch #"); axes[0].set_ylabel("Count")
axes[1].bar(category_df["category"], category_df["total_spend"])
axes[1].set_title("Running spend by category (Rs)"); axes[1].tick_params(axis="x", rotation=45)
rule_counts = unique_alerts["rule"].value_counts()
axes[2].bar(rule_counts.index, rule_counts.values, color="tab:red")
axes[2].set_title("Unique fraud alerts by rule"); axes[2].tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.show()

### ✅ Case study — what you should see
| Rule | Account | First fires | Why it fired |
|------|---------|-------------|--------------|
| `HIGH_VALUE` | ACC1003 (Meera) | batch 4 | ₹1,25,000 purchase at Croma |
| `LIMIT_BREACH` | ACC1003 (Meera) | batch 4 | that purchase pushes her running spend above her ₹1,00,000 limit — fires from **state** and keeps firing every batch |
| `VELOCITY` | ACC1005 (Sara) | batch 6 | 7 tiny ₹49 online payments across batches 6–7 → classic **card testing** |
| `GEO_ANOMALY` | ACC1001 (Diya) | batch 9 | ATM withdrawals in Mumbai and Chennai (plus home city Delhi), then Hyderabad, within 6 s |
| `VELOCITY` | ACC1001 (Diya) | batch 9 | the same ATM burst is also 5 transactions in 6 s → **one incident can trigger several rules** |

Also notice: **3 malformed records** were rejected without crashing the stream, and the same alert re-appears in consecutive
windows — real systems de-duplicate alerts (we did it with `drop_duplicates`; in production, a state store or an alert DB does it).

---
## 7. Bonus: The Same Case Study in Structured Streaming (the modern API)

DStreams are the **legacy** API (deprecated since Spark 3.4). New projects use **Structured Streaming**: the stream is an
*unbounded DataFrame*, you write normal DataFrame/SQL code, and you get **event-time windows**, **watermarks** and
**exactly-once sinks**. Here we drop each batch as a CSV file into a folder and let Spark pick them up.

In [ ]:
# ------------------------------------------------------------
# BONUS STEP 1 : Write each micro-batch as a file into a "landing" folder
# ------------------------------------------------------------
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

LANDING_DIR = os.path.join(BASE_DIR, "landing_zone")
SS_CHECKPOINT = os.path.join(BASE_DIR, "ss_checkpoints")
shutil.rmtree(LANDING_DIR, ignore_errors=True); shutil.rmtree(SS_CHECKPOINT, ignore_errors=True)
os.makedirs(LANDING_DIR)

for i, batch in enumerate(txn_batches):
    with open(os.path.join(LANDING_DIR, f"txns_{i:03d}.csv"), "w") as f:
        f.write("\n".join(batch) + "\n")
print(len(os.listdir(LANDING_DIR)), "files written to", LANDING_DIR)

txn_schema = StructType([
    StructField("txn_id", StringType()),     StructField("event_time", StringType()),
    StructField("account_id", StringType()), StructField("merchant", StringType()),
    StructField("amount", DoubleType()),     StructField("city", StringType()),
    StructField("channel", StringType()),
])

In [ ]:
# ------------------------------------------------------------
# BONUS STEP 2 : Streaming DataFrame + three queries
# ------------------------------------------------------------
merchant_df = spark.createDataFrame(list(MERCHANT_CATEGORY.items()), ["merchant", "category"])

stream_df = (spark.readStream
             .schema(txn_schema)                    # streaming file sources need an explicit schema
             .option("maxFilesPerTrigger", 4)       # 4 files (= 4 DStream batches) per micro-batch
             .csv(LANDING_DIR))

clean_df = (stream_df
            .filter(F.col("account_id").isNotNull() & (F.col("amount") > 0))   # drop bad records
            .withColumn("event_time", F.to_timestamp("event_time"))
            .join(merchant_df, "merchant", "left"))                            # stream-static join

# Query 1: event-time tumbling window of 10 s -> spend per category
category_window = (clean_df
    .groupBy(F.window("event_time", "10 seconds"), "category")
    .agg(F.count("*").alias("txns"), F.round(F.sum("amount"), 2).alias("total_amount")))

# Query 2: VELOCITY rule with an event-time SLIDING window (6 s window, 2 s slide)
velocity = (clean_df
    .groupBy(F.window("event_time", "6 seconds", "2 seconds"), "account_id")
    .count()
    .filter(F.col("count") >= VELOCITY_THRESHOLD))

# Query 3: HIGH_VALUE rule - a simple stateless filter, append mode
high_value = clean_df.filter(F.col("amount") >= HIGH_VALUE_THRESHOLD) \
                     .select("txn_id", "event_time", "account_id", "merchant", "amount", "city")

queries = []
for name, df, mode in [("ss_category_window", category_window, "complete"),
                       ("ss_velocity",        velocity,        "complete"),
                       ("ss_high_value",      high_value,      "append")]:
    q = (df.writeStream
           .format("memory")                        # in-memory table, queryable with spark.sql (demo sink)
           .queryName(name)
           .outputMode(mode)
           .option("checkpointLocation", os.path.join(SS_CHECKPOINT, name))
           .trigger(availableNow=True)              # process everything available, then stop
           .start())
    queries.append(q)

for q in queries:
    q.awaitTermination()
print("All Structured Streaming queries finished ✔")

In [ ]:
# ------------------------------------------------------------
# BONUS STEP 3 : Query the results with SQL
# ------------------------------------------------------------
print("Spend per category per 10-second event-time window")
spark.sql("""SELECT date_format(window.start, 'HH:mm:ss') AS win_start,
                    date_format(window.end,   'HH:mm:ss') AS win_end,
                    category, txns, total_amount
             FROM ss_category_window ORDER BY win_start, category""").show(30, truncate=False)

print("VELOCITY alerts (Structured Streaming)")
spark.sql("""SELECT date_format(window.start, 'HH:mm:ss') AS win_start,
                    date_format(window.end,   'HH:mm:ss') AS win_end, account_id, count
             FROM ss_velocity ORDER BY win_start""").show(truncate=False)

print("HIGH_VALUE alerts (Structured Streaming)")
spark.sql("SELECT * FROM ss_high_value").show(truncate=False)

### DStreams vs Structured Streaming
| Aspect | DStreams (Sections 1–6) | Structured Streaming (Section 7) |
|--------|-------------------------|----------------------------------|
| Abstraction | Sequence of RDDs | Unbounded DataFrame / table |
| Windows based on | **Processing time** (when data arrived) | **Event time** (when it happened) + watermarks for late data |
| Optimizer | None (you hand-tune RDD code) | Catalyst + Tungsten |
| Guarantees | At-least-once by default | Exactly-once with replayable source + idempotent sink |
| Status | Legacy / deprecated | Recommended for all new work |

---
## 8. 🧪 Practice Exercises
1. **Batch interval:** Re-run Demo 4.1 with `WINDOW, SLIDE = 4, 2`. Predict the output before running.
2. **Top-N per batch:** In Demo 3.3, use `transform` to return only the **most frequent error code** in each batch.
3. **New fraud rule:** Add a `NIGHT_ATM` rule — ATM withdrawals above ₹9,000 (hint: filter on `channel == "ATM"`).
4. **Alert de-duplication in state:** Write an `updateStateByKey` function keyed by `(account_id, rule)` that emits an alert only the **first** time it is seen.
5. **Structured Streaming:** Add a `GEO_ANOMALY` query using `F.approx_count_distinct("city")` over a 6-second sliding window.

In [ ]:
# ------------------------------------------------------------
# CLEAN-UP : stop any active streaming context and Spark
# ------------------------------------------------------------
active = StreamingContext.getActive()
if active is not None:
    active.stop(stopSparkContext=False, stopGraceFully=False)
spark.stop()
print("Spark stopped. 🎉 Lab complete!")